# Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local **agentic workflow** for English-to-Chinese cross-lingual dialogue summarization.

The proposed pipeline is:

```text
English Dialogue
→ Agent 1: English Summarization
→ Agent 2: Chinese Translation
→ Agent 3: Evaluation
→ Agent 4: Revision
→ Agent 5: Final Selection
→ Final Chinese Summary
```

The agents are controlled in this notebook. Ollama only serves the local models.


## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the models locally.

### Recommended model setup

```bash
ollama pull qwen3.5:9b
ollama pull translategemma:12b
```

If `translategemma:12b` is too slow on your machine, use:

```bash
ollama pull translategemma:4b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [51]:
# Cell 1: Install required Python packages
# Run this only once if the packages are not installed.

!pip install requests pandas tqdm


The folder you are executing pip from can no longer be found.


In [29]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

MAIN_MODEL = "qwen3.5:9b"
TRANSLATION_MODEL = "translategemma:12b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

DATA_PATH = Path("data/processed/test_sample.jsonl")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_OUTPUT_PATH = OUTPUT_DIR / "two_agent_outputs.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "two_agent_final_summaries.csv"

In [30]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])
        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")
        return True
    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))
        return False

_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['translategemma:12b', 'qwen3.5:9b']


In [31]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 600,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [ ]:
# Cell 5: Prompt templates

SUMMARIZATION_PROMPT = """You are an English dialogue summarization agent.

Your task is to summarize the given English dialogue into a concise English summary.

Requirements:
- Preserve the main intent, key events, decisions, and important entities.
- Do not add information that is not supported by the dialogue.
- Do not include minor details unless they are necessary for understanding the main point.
- Keep the summary concise.
- Use clear and natural English.
- Output only the English summary.

Input dialogue:
{dialogue}

English summary:
"""

TRANSLATION_PROMPT = """You are a professional English-to-Simplified-Chinese translation agent.

Your task is to translate the given English summary into natural Simplified Chinese.

Requirements:
- Preserve the original meaning accurately.
- Preserve names, numbers, dates, and important entities.
- Use Chinese transliterations for common English names.
- Do not include the original English names in parentheses unless necessary.
- Do not add new information.
- Do not remove important information.
- Use fluent and natural Simplified Chinese.
- Output only the Chinese translation.

English summary:
{english_summary}

Chinese summary:
"""

In [33]:
# Cell 6: Utility functions for JSONL storage

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)
    return {str(record["id"]) for record in records if "id" in record}

In [34]:
# Cell 7: Agent functions

def summarization_agent(dialogue: str) -> str:
    prompt = SUMMARIZATION_PROMPT.format(dialogue=dialogue)
    return call_ollama(
        model=MAIN_MODEL,
        prompt=prompt,
        temperature=0.2,
    )


def translation_agent(english_summary: str) -> str:
    prompt = TRANSLATION_PROMPT.format(english_summary=english_summary)
    return call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [35]:
# Cell 8: Two-agent pipeline

def run_two_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_summary = (
        example.get("reference_chinese_summary")
        or example.get("summary")
        or ""
    )

    english_summary = summarization_agent(dialogue)
    chinese_summary = translation_agent(english_summary)

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_chinese_summary": reference_summary,
        "agent1_english_summary": english_summary,
        "agent2_chinese_summary": chinese_summary,
        "final_summary": chinese_summary,
    }

## 3. Test with One Example

Start with one example before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [36]:
# Cell 9: Define one test example

example = {
    "id": "sample_001",
    "dialogue": """Amanda: Are we still meeting tomorrow morning?
Brian: I don't think I can make it. I have a doctor's appointment at 10.
Amanda: No problem. Would Friday afternoon work instead?
Brian: Yes, Friday after 2 works for me.
Amanda: Great, let's meet at 2:30 on Friday.""",
    "reference_chinese_summary": "阿曼达和布莱恩讨论了重新安排会议，因为布莱恩明天上午有医生预约。他们同意周五下午2:30见面。"
}

example

{'id': 'sample_001',
 'dialogue': "Amanda: Are we still meeting tomorrow morning?\nBrian: I don't think I can make it. I have a doctor's appointment at 10.\nAmanda: No problem. Would Friday afternoon work instead?\nBrian: Yes, Friday after 2 works for me.\nAmanda: Great, let's meet at 2:30 on Friday.",
 'reference_chinese_summary': '阿曼达和布莱恩讨论了重新安排会议，因为布莱恩明天上午有医生预约。他们同意周五下午2:30见面。'}

In [37]:
# Cell 10: Run the two-agent pipeline for one example

result = run_two_agent_pipeline(example)
result

{'id': 'sample_001',
 'dialogue': "Amanda: Are we still meeting tomorrow morning?\nBrian: I don't think I can make it. I have a doctor's appointment at 10.\nAmanda: No problem. Would Friday afternoon work instead?\nBrian: Yes, Friday after 2 works for me.\nAmanda: Great, let's meet at 2:30 on Friday.",
 'reference_chinese_summary': '阿曼达和布莱恩讨论了重新安排会议，因为布莱恩明天上午有医生预约。他们同意周五下午2:30见面。',
 'agent1_english_summary': "Amanda and Brian rescheduled their meeting from tomorrow morning to Friday afternoon at 2:30 PM due to Brian's doctor's appointment.",
 'agent2_chinese_summary': '由于布莱恩（Brian）需要去看医生，艾曼达（Amanda）和布莱恩将他们原定于明天上午的会议改期至星期五下午2点30分。',
 'final_summary': '由于布莱恩（Brian）需要去看医生，艾曼达（Amanda）和布莱恩将他们原定于明天上午的会议改期至星期五下午2点30分。'}

In [38]:
# Cell 11: Print two-agent pipeline result clearly

def print_two_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: English Summary ===")
    print(result["agent1_english_summary"])
    print()

    print("=== Agent 2: Chinese Summary ===")
    print(result["agent2_chinese_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])


print_two_agent_result(result)

=== Original Dialogue ===
Amanda: Are we still meeting tomorrow morning?
Brian: I don't think I can make it. I have a doctor's appointment at 10.
Amanda: No problem. Would Friday afternoon work instead?
Brian: Yes, Friday after 2 works for me.
Amanda: Great, let's meet at 2:30 on Friday.

=== Agent 1: English Summary ===
Amanda and Brian rescheduled their meeting from tomorrow morning to Friday afternoon at 2:30 PM due to Brian's doctor's appointment.

=== Agent 2: Chinese Summary ===
由于布莱恩（Brian）需要去看医生，艾曼达（Amanda）和布莱恩将他们原定于明天上午的会议改期至星期五下午2点30分。

=== Reference Chinese Summary ===
阿曼达和布莱恩讨论了重新安排会议，因为布莱恩明天上午有医生预约。他们同意周五下午2:30见面。

=== Final Chinese Summary ===
由于布莱恩（Brian）需要去看医生，艾曼达（Amanda）和布莱恩将他们原定于明天上午的会议改期至星期五下午2点30分。


## 4. Save One Result

This saves all intermediate outputs and the final output.


In [40]:
# Cell 12: Save the single-example result to JSONL

append_jsonl(result, FULL_OUTPUT_PATH)
print(f"Saved to: {FULL_OUTPUT_PATH}")

Saved to: results/two_agent_outputs.jsonl


In [45]:
# 12-1
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Output file:", (Path.cwd() / "results/two_agent_outputs.jsonl").resolve())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents
Output file: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/two_agent_outputs.jsonl


In [50]:
# 12-2: Reset output files
FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)

error_path = OUTPUT_DIR / "two_agent_errors.jsonl"
error_path.unlink(missing_ok=True)

print("Output files reset.")

Output files reset.


## 5. Load a Test Dataset

Expected JSONL format:

```json
{"id": "sample_001", "dialogue": "...", "summary": "..."}
{"id": "sample_002", "dialogue": "...", "summary": "..."}
```

The `summary` field is optional, but recommended for evaluation.


In [47]:
# Cell 13: Load dataset from JSONL

def load_dataset(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        print(f"Dataset not found: {path}")
        print("Using the single example instead.")
        return [example]
    return load_jsonl(path)


test_data = load_dataset(DATA_PATH)
print(f"Loaded {len(test_data)} examples.")
print(test_data[0])

Dataset not found: data/processed/test_sample.jsonl
Using the single example instead.
Loaded 1 examples.
{'id': 'sample_001', 'dialogue': "Amanda: Are we still meeting tomorrow morning?\nBrian: I don't think I can make it. I have a doctor's appointment at 10.\nAmanda: No problem. Would Friday afternoon work instead?\nBrian: Yes, Friday after 2 works for me.\nAmanda: Great, let's meet at 2:30 on Friday.", 'reference_chinese_summary': '阿曼达和布莱恩讨论了重新安排会议，因为布莱恩明天上午有医生预约。他们同意周五下午2:30见面。'}


## 6. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [48]:
# Cell 14: Batch inference with two-agent pipeline

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running two-agent pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_two_agent_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }
        append_jsonl(error_record, OUTPUT_DIR / "two_agent_errors.jsonl")
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running two-agent pipeline:   0%|          | 0/1 [00:00<?, ?it/s]

Finished. Outputs saved to: results/two_agent_outputs.jsonl


## 7. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [49]:
# Cell 15: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []
for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_summary": record.get("reference_chinese_summary", ""),
        "agent1_english_summary": record.get("agent1_english_summary", ""),
        "agent2_chinese_summary": record.get("agent2_chinese_summary", ""),
    })

df = pd.DataFrame(rows)
df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df.head()

Saved final summaries to: results/two_agent_final_summaries.csv


,id,final_summary,reference_summary,agent1_english_summary,agent2_chinese_summary
0,sample_001,由于布莱恩（Brian）需要去看医生，艾曼达（Amanda）和布莱恩将他们原定于明天上午的会...,阿曼达和布莱恩讨论了重新安排会议，因为布莱恩明天上午有医生预约。他们同意周五下午2:30见面。,Amanda and Brian rescheduled their meeting fro...,由于布莱恩（Brian）需要去看医生，艾曼达（Amanda）和布莱恩将他们原定于明天上午的会...


## 8. Next Steps

Recommended experimental plan:

1. Run 5 examples and inspect every intermediate output manually.
2. Run 20 examples and check whether JSON outputs are stable.
3. Run 50–100 examples for preliminary comparison.
4. Compare against the existing BART/mBART pipeline.
5. Add ROUGE-L, BERTScore, and OmniScore evaluation.
6. If branching becomes complex, migrate the workflow to LangGraph.
